# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Title:** Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
- **Identifier:** 10.71728/senscience.vq4a-28xa
- **Description:** This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes various fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside calculated parameters such as pI and normalized abundances across different samples.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n")
print(f"Dataset description: {metadata.description}\n")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")
print(f"Published on: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Print all available record sets with their @ids and field structure
print("Available Record Sets and their @ids:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- Record set: {rs['@id']} (name: {rs['name'] if 'name' in rs else 'N/A'})")
    fields = rs.get('field', [])
    # field can be a dict, list of dicts, or single field
    field_ids = []
    if isinstance(fields, dict):
        field_ids = [fields['@id']]
    elif isinstance(fields, list):
        for f in fields:
            if isinstance(f, dict) and '@id' in f:
                field_ids.append(f['@id'])
            elif isinstance(f, str):
                field_ids.append(f)
    elif isinstance(fields, str):
        field_ids = [fields]
    print(f"    Fields: {field_ids}")
    
print("\nExample records from the first record set:")
if record_sets:
    sample_rs_id = record_sets[0]['@id']
    for i, record in enumerate(dataset.records(record_set=sample_rs_id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract all dataframes indexed by record set @id
rs_ids = [rs['@id'] for rs in record_sets]
dataframes = {}
for record_set_id in rs_ids:
    print(f"Loading data for record set: {record_set_id}")
    recs = list(dataset.records(record_set=record_set_id))
    if recs:
        dataframes[record_set_id] = pd.DataFrame(recs)
    else:
        dataframes[record_set_id] = pd.DataFrame()

# Example: Show columns and head of first non-empty record set
first_df_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        first_df_id = rsid
        print(f"\nColumns in record set {first_df_id}:")
        print(df.columns.tolist())
        display(df.head())
        break
if not first_df_id:
    print("No non-empty record set data available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes to prepare it for further analysis.


In [ ]:
# For demonstration, pick the first non-empty DataFrame and perform EDA
import numpy as np

if first_df_id:
    df = dataframes[first_df_id]

    # Try to auto-detect likely numeric columns
    # If there is a column 'coverage_percent' or 'MW', use it; otherwise use the first numeric-looking column
    numeric_candidates = [
        c for c in df.columns if df[c].dtype.kind in 'biufc' and not all(df[c].isna())
    ]
    if not numeric_candidates:
        # Try to coerce
        for c in df.columns:
            try:
                df[c+'_coerced'] = pd.to_numeric(df[c], errors='coerce')
                if df[c+'_coerced'].notna().sum() > 0:
                    numeric_candidates.append(c+'_coerced')
                    break
            except Exception:
                continue

    # Use first numeric field for demonstration
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
    else:
        print('No numeric field detected for analysis.')
        numeric_field = None

    if numeric_field:
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].dropna().quantile(0.75)  # Top 25% as example threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / (filtered_df[numeric_field].std())

        print(f"\nFirst few normalized values of {numeric_field}:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try group by a likely categorical field (select any field not numeric and has <20 unique values)
        group_field = None
        for c in df.columns:
            if (df[c].dtype == object) and (df[c].nunique() < 20) and (c != numeric_field):
                group_field = c
                break
        
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean of {numeric_field} by {group_field}:\n")
            display(grouped_df.head())
        else:
            print("No suitable categorical field for grouping detected.")
    else:
        print("No numeric field available for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if first_df_id and numeric_field and not filtered_df.empty:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field], kde=True, bins=20)
    plt.title(f'Distribution of {numeric_field} (filtered)')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If a group_field is available, make a boxplot
    if 'group_field' in locals() and group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.xticks(rotation=45)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()
else:
    print('Not enough numeric data to plot.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded and explored the FAIR^2 Croissant dataset using `mlcroissant`.
- Explored record sets and demonstrated data extraction, filtering, normalization, and grouping using field `@id` references.
- Created simple visualizations to understand the distribution and group-wise statistics of numeric fields.
- This reproducible, standards-based approach can be adapted for any dataset with a proper Croissant schema.
